# Stock Trend Prediction and Backtesting

A compact, hands-on notebook using ~10 years of AAPL data.
Build simple technical features, try a few classical ML models, and backtest a basic long-only rule.
Designed for learning — results are illustrative, not investment advice.

Follow cells top-to-bottom; each section has a short note.

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from sklearn.model_selection import train_test_split, GridSearchCV, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, roc_curve

from src import utils, indicators, backtest

pd.options.display.max_columns = 50
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (11,5)


## Presentation: Key results

Below are three quick visuals to show during a demo.

Talk track (short):
- One line on the data used (AAPL, 10 years).
- One line on features (moving averages, RSI, MACD, volatility).
- One line on modeling (tried a few classical models; Random Forest tuned).
- One line on results (modest accuracy; strategy didn't beat buy-and-hold after costs).

In [ ]:
from IPython.display import Image, display
display(Image('images/model_comparison.png'))
display(Image('images/feature_importances.png'))
display(Image('images/cumulative_returns.png'))


## Download data

In [ ]:
ticker = 'AAPL'
start = '2016-01-01'
end = '2026-01-01'

df = utils.download_data(ticker, start, end)
df = utils.clean_data(df)
print(f"Data range: {df.index.min().date()} to {df.index.max().date()}, rows={len(df)}")
df.head()


## Data checks

In [ ]:
display(df.info())
display(df.describe().round(4))
print('Missing values:', df.isna().sum())
print('Duplicate index?', df.index.duplicated().any())


## 4. EDA : quick visuals: price, volume, returns, volatility, distributions

In [ ]:
# Adj Close
plt.figure(figsize=(12,4))
plt.plot(df.index, df['Adj Close'])
plt.title('AAPL Adjusted Close Price')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.show()


In [ ]:
# Volume
plt.figure(figsize=(12,3))
plt.plot(df['Volume'])
plt.title('Daily Volume')
plt.show()


In [ ]:
# Returns and KDE
df_r = indicators.calculate_returns(df)
plt.figure(figsize=(10,4))
sns.histplot(df_r['Return'].dropna(), bins=80, kde=True)
plt.title('Histogram and KDE of Daily Returns')
plt.show()


In [ ]:
# Rolling volatility (20-day)
vol20 = indicators.calculate_volatility(df['Adj Close'], 20)
plt.figure(figsize=(12,4))
plt.plot(vol20)
plt.title('20-day Rolling Volatility (annualized)')
plt.show()


In [ ]:
# Boxplot of returns by year
tmp = df_r.copy()
tmp['Year'] = tmp.index.year
plt.figure(figsize=(12,5))
sns.boxplot(x='Year', y='Return', data=tmp.reset_index())
plt.title('Daily Return Distribution by Year')
plt.xticks(rotation=45)
plt.show()


In [ ]:
# Correlation heatmap of selected features
plt.figure(figsize=(8,6))
sns.heatmap(df[['Open','High','Low','Close','Adj Close','Volume']].corr(), annot=True, fmt='.2f')
plt.title('Correlation Matrix')
plt.show()


## Feature engineering

In [ ]:
df_fe = df.copy()
# basic returns
df_fe['Return'] = df_fe['Adj Close'].pct_change()
df_fe['LogReturn'] = np.log(df_fe['Adj Close']).diff()
# moving averages
df_fe['MA10'] = indicators.moving_average(df_fe['Adj Close'], 10)
df_fe['MA20'] = indicators.moving_average(df_fe['Adj Close'], 20)
df_fe['MA50'] = indicators.moving_average(df_fe['Adj Close'], 50)
# EMA and MACD
df_fe['EMA12'] = df_fe['Adj Close'].ewm(span=12, adjust=False).mean()
df_fe['EMA26'] = df_fe['Adj Close'].ewm(span=26, adjust=False).mean()
df_fe['MACD'] = df_fe['EMA12'] - df_fe['EMA26']
# Volatility and RSI
df_fe['Volatility20'] = indicators.calculate_volatility(df_fe['Adj Close'], 20)
df_fe['RSI14'] = indicators.calculate_rsi(df_fe['Adj Close'], 14)
# Price range and volume change
df_fe['PriceRange'] = (df_fe['High'] - df_fe['Low']) / df_fe['Low']
df_fe['VolumeChange'] = df_fe['Volume'].pct_change()
# lag features (1-day and 2-day returns)
df_fe['Lag1'] = df_fe['Return'].shift(1)
df_fe['Lag2'] = df_fe['Return'].shift(2)
# ratios
df_fe['CloseToMA10'] = df_fe['Adj Close'] / df_fe['MA10'] - 1

# drop na rows produced by indicators
df_fe = df_fe.dropna()

# quick peek
print(df_fe.iloc[-3:][['Adj Close','Return','MA10','RSI14']])


Notes: a quick intuition for these features:
- EMA (exponential MA): gives more weight to recent prices, so it reacts faster.
- MACD (EMA12 - EMA26): a simple momentum indicator showing short vs long EMA.
- Lag features (Lag1, Lag2): previous-day returns that can capture short-term autocorrelation.
- Ratios (CloseToMA10): normalize price relative to a moving average so values are scale-free.

## Target

In [ ]:
df_fe['Target'] = (df_fe['Adj Close'].shift(-1) > df_fe['Adj Close']).astype(int)
df_fe = df_fe.dropna()
print('Target balance:')
print(df_fe['Target'].value_counts(normalize=True).round(3))


## Prepare dataset

In [ ]:
feature_cols = ['Return','LogReturn','MA10','MA20','MA50','EMA12','EMA26','MACD','Volatility20','RSI14','PriceRange','VolumeChange','Lag1','Lag2','CloseToMA10']
X, y = utils.prepare_dataset(df_fe, feature_cols, 'Target')
# time-aware train/test split (keep order)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
print('Train/test sizes:', X_train.shape, X_test.shape)
scaler = StandardScaler()
X_train_s = pd.DataFrame(scaler.fit_transform(X_train), index=X_train.index, columns=X_train.columns)
X_test_s = pd.DataFrame(scaler.transform(X_test), index=X_test.index, columns=X_test.columns)


## Modeling

In [ ]:
# Train baseline models: quick comparison to pick a candidate (kept simple for clarity)
models = {
    'LogisticRegression': LogisticRegression(max_iter=500),
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(probability=True, kernel='rbf')
}
results = []
fitted = {}
for name, m in models.items():
    # fit on training data and collect test metrics
    m.fit(X_train_s, y_train)
    fitted[name] = m
    preds = m.predict(X_test_s)
    probs = m.predict_proba(X_test_s)[:,1] if hasattr(m, 'predict_proba') else m.decision_function(X_test_s)
    res = {
        'Model': name,
        'Accuracy': accuracy_score(y_test, preds),
        'Precision': precision_score(y_test, preds, zero_division=0),
        'Recall': recall_score(y_test, preds, zero_division=0),
        'F1': f1_score(y_test, preds, zero_division=0),
        'ROC_AUC': roc_auc_score(y_test, probs)
    }
    results.append(res)
results_df = pd.DataFrame(results).set_index('Model')
display(results_df.round(3))


In [ ]:
# confusion matrix and classification report for best model (RandomForest as example)
best = fitted['RandomForest']
preds_rf = best.predict(X_test_s)
probs_rf = best.predict_proba(X_test_s)[:,1]
print('Classification report (RandomForest):')
print(classification_report(y_test, preds_rf))
cm = confusion_matrix(y_test, preds_rf)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

# ROC curve
fpr, tpr, _ = roc_curve(y_test, probs_rf)
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f'RF AUC={roc_auc_score(y_test, probs_rf):.3f}')
plt.plot([0,1],[0,1],'--',color='grey')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()


## Cross-validation

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
scores = []
for train_idx, val_idx in tscv.split(X_train_s):
    Xtr, Xv = X_train_s.iloc[train_idx], X_train_s.iloc[val_idx]
    ytr, yv = y_train.iloc[train_idx], y_train.iloc[val_idx]
    m = RandomForestClassifier(n_estimators=50, random_state=42)
    m.fit(Xtr, ytr)
    preds = m.predict(Xv)
    scores.append(f1_score(yv, preds))
print('TimeSeriesSplit F1 scores:', [round(s,3) for s in scores], 'mean=', round(np.mean(scores),3))


## Hyperparameter tuning

In [ ]:
param_grid = {'n_estimators':[50,100],'max_depth':[3,6,None],'min_samples_split':[2,5]}
grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, scoring='f1', n_jobs=-1)
grid.fit(X_train_s, y_train)
print('Best params:', grid.best_params_)
best_rf = grid.best_estimator_
print('Test F1:', round(f1_score(y_test, best_rf.predict(X_test_s)),3))


## Feature importance

In [ ]:
imp = pd.Series(best_rf.feature_importances_, index=X_train.columns).sort_values()
plt.figure(figsize=(8,6))
imp.plot(kind='barh')
plt.title('Feature Importances (Random Forest)')
plt.show()


## Backtesting

In [ ]:
# predict on test set and build positions
# Using a simple next-day execution assumption: signals are generated at close and applied next day
preds_test = pd.Series(best_rf.predict(X_test_s), index=X_test_s.index)
positions = backtest.generate_signals(preds_test)
# price series for test period
price_test = df_fe['Adj Close'].reindex(X_test_s.index)
# calculate returns with a small transaction cost (0.1%) and 100% position when long
bt = backtest.calculate_returns(price_test, positions, position_size=1.0, transaction_cost=0.001)
# cumulative returns for plotting
bt['CumMarket'] = (1 + bt['MarketReturn']).cumprod() - 1
bt['CumStrat'] = (1 + bt['StrategyReturn']).cumprod() - 1
plt.figure(figsize=(12,6))
plt.plot(bt.index, bt['CumMarket'], label='Market')
plt.plot(bt.index, bt['CumStrat'], label='Strategy')
plt.legend()
plt.title('Cumulative Returns (with transaction cost = 0.1%)')
plt.show()


In [ ]:
# drawdown and performance
bt['Drawdown'] = backtest.calculate_drawdown((1 + bt['StrategyReturn']).cumprod())
plt.figure(figsize=(12,4))
plt.plot(bt['Drawdown'])
plt.title('Strategy Drawdown')
plt.show()
perf = backtest.performance_summary(bt)
import pprint
pprint.pprint(perf)


## Sanity checks

In [ ]:
# check win rate of strategy (days with positive strategy return)
win_rate = (bt['StrategyReturn'] > 0).mean()
print('Strategy win rate:', round(win_rate,3))
# effect of transaction cost: compute performance with no costs vs with costs
bt_nocost = backtest.calculate_returns(price_test, positions, position_size=1.0, transaction_cost=0.0)
bt_nocost['CumStrat'] = (1 + bt_nocost['StrategyReturn']).cumprod() - 1
print('Final strategy return without cost:', round(bt_nocost['CumStrat'].iloc[-1],3))
print('Final strategy return with cost:', round(bt['CumStrat'].iloc[-1],3))


## Conclusion

Summary: I made a set of simple technical features, trained a few standard models, tuned a Random Forest, and backtested a basic long-only rule. The goal was to learn the pipeline — the results are modest and mainly for discussion.

Limitations: only one ticker, a small hand-crafted feature set, no realistic execution (no slippage or market impact), and potential leakage if features are not time-aligned.

Next steps: try multiple tickers, include transaction costs and slippage, and use walk-forward/time-series cross-validation for more reliable validation.